# Semana 7: Gradient Boosting (XGBoost, LightGBM)
## Notebook Conceptual (NB1) – Boosting desde Cero y Experimentación

**Propósito:** Introducir el concepto de boosting secuencial y las implementaciones modernas eficientes, mostrando cómo ajustan errores residuales.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Implementar un gradient boosting simple desde cero para regresión.
- Visualizar cómo las predicciones mejoran iterativamente.
- Usar XGBoost y LightGBM y variar hiperparámetros (learning_rate, n_estimators, max_depth).
- Observar early stopping en acción.
- Comparar tiempos de entrenamiento entre XGBoost y Random Forest.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# XGBoost y LightGBM
import xgboost as xgb
import lightgbm as lgb

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Generación de Datos Sintéticos para Regresión

Creamos datos con una relación no lineal (función seno) más ruido para visualizar claramente la mejora iterativa del boosting.

In [ ]:
# Función verdadera
def true_function(x):
    return np.sin(x) * 5 + x/2

# Generamos datos
n_samples = 300
X = np.random.uniform(-5, 5, n_samples).reshape(-1, 1)
y = true_function(X.ravel()) + np.random.randn(n_samples) * 1.5

# Ordenamos para visualización
X_sorted = np.sort(X, axis=0)
y_true_sorted = true_function(X_sorted.ravel())

# Dividimos en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.6, label='Train')
plt.scatter(X_test, y_test, alpha=0.6, label='Test')
plt.plot(X_sorted, y_true_sorted, 'r-', linewidth=2, label='Función verdadera')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos sintéticos para regresión')
plt.legend()
plt.grid(True)
plt.show()

---
## 2. Implementación Manual de Gradient Boosting

Implementamos un gradient boosting simple:
- Inicializamos con la media de y.
- En cada iteración, calculamos los residuos (errores) y ajustamos un árbol pequeño a esos residuos.
- Actualizamos la predicción sumando el árbol escalado por learning_rate.

**Nota:** Usamos árboles de regresión poco profundos (max_depth=3).

In [ ]:
class GradientBoostingManual:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.initial_pred = None
    
    def fit(self, X, y):
        # Inicialización: predicción constante (media de y)
        self.initial_pred = np.mean(y)
        current_pred = np.full_like(y, self.initial_pred, dtype=float)
        
        for i in range(self.n_estimators):
            # Calcular residuos (gradiente negativo para MSE)
            residuals = y - current_pred
            
            # Entrenar árbol sobre los residuos
            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=42+i)
            tree.fit(X, residuals)
            self.trees.append(tree)
            
            # Actualizar predicciones
            current_pred += self.learning_rate * tree.predict(X)
    
    def predict(self, X):
        pred = np.full(X.shape[0], self.initial_pred)
        for tree in self.trees:
            pred += self.learning_rate * tree.predict(X)
        return pred

# Entrenamos el modelo manual
gb_manual = GradientBoostingManual(n_estimators=50, learning_rate=0.1, max_depth=3)
gb_manual.fit(X_train, y_train.ravel())

# Predicciones
y_pred_train = gb_manual.predict(X_train)
y_pred_test = gb_manual.predict(X_test)

print(f"MSE Train (manual): {mean_squared_error(y_train, y_pred_train):.4f}")
print(f"MSE Test (manual): {mean_squared_error(y_test, y_pred_test):.4f}")

## 3. Visualización de la Mejora Iterativa

Mostramos cómo la predicción se acerca a los valores reales a medida que se añaden árboles.

In [ ]:
# Función para predecir con un número limitado de árboles
def predict_with_n_trees(gb_model, X, n_trees):
    pred = np.full(X.shape[0], gb_model.initial_pred)
    for i in range(n_trees):
        pred += gb_model.learning_rate * gb_model.trees[i].predict(X)
    return pred

# Evaluamos en test con diferentes números de árboles
tree_counts = [1, 5, 10, 20, 30, 40, 50]
X_plot = np.linspace(-5, 5, 300).reshape(-1, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i, n in enumerate(tree_counts):
    y_plot = predict_with_n_trees(gb_manual, X_plot, n)
    axes[i].scatter(X_test, y_test, alpha=0.3, label='Datos test')
    axes[i].plot(X_plot, y_plot, 'r-', linewidth=2, label=f'Predicción ({n} árboles)')
    axes[i].plot(X_sorted, y_true_sorted, 'g--', linewidth=1, label='Función verdadera')
    axes[i].set_title(f'{n} árboles')
    axes[i].set_xlabel('X')
    axes[i].set_ylabel('y')
    axes[i].legend()
    axes[i].grid(True)

# Eliminar el último subplot vacío
for i in range(len(tree_counts), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 4. Evolución del Error con el Número de Árboles

In [ ]:
train_errors = []
test_errors = []

for n in range(1, 51):
    y_pred_train_n = predict_with_n_trees(gb_manual, X_train, n)
    y_pred_test_n = predict_with_n_trees(gb_manual, X_test, n)
    train_errors.append(mean_squared_error(y_train, y_pred_train_n))
    test_errors.append(mean_squared_error(y_test, y_pred_test_n))

plt.figure(figsize=(10, 5))
plt.plot(range(1, 51), train_errors, 'b-', label='Train MSE')
plt.plot(range(1, 51), test_errors, 'r-', label='Test MSE')
plt.xlabel('Número de árboles')
plt.ylabel('MSE')
plt.title('Evolución del error con el número de árboles')
plt.legend()
plt.grid(True)
plt.show()

---
## 5. XGBoost: Experimentación con Hiperparámetros

Usamos XGBoost y variamos learning_rate, n_estimators y max_depth.

In [ ]:
# Convertimos datos a formato DMatrix (opcional pero óptimo)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Configuración base
params_base = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': 42
}

# Probamos diferentes learning rates
learning_rates = [0.01, 0.1, 0.3]
results_lr = []

for lr in learning_rates:
    params = params_base.copy()
    params['learning_rate'] = lr
    model = xgb.train(params, dtrain, num_boost_round=100, evals=[(dtest, 'test')],
                      early_stopping_rounds=10, verbose_eval=False)
    y_pred = model.predict(dtest)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results_lr.append({'learning_rate': lr, 'RMSE': rmse, 'best_iter': model.best_iteration})

df_lr = pd.DataFrame(results_lr)
print("=== Efecto de learning_rate ===")
df_lr

In [ ]:
# Probamos diferentes max_depth
max_depths = [2, 3, 5, 8]
results_depth = []

for depth in max_depths:
    params = params_base.copy()
    params['learning_rate'] = 0.1
    params['max_depth'] = depth
    model = xgb.train(params, dtrain, num_boost_round=100, evals=[(dtest, 'test')],
                      early_stopping_rounds=10, verbose_eval=False)
    y_pred = model.predict(dtest)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results_depth.append({'max_depth': depth, 'RMSE': rmse, 'best_iter': model.best_iteration})

df_depth = pd.DataFrame(results_depth)
print("=== Efecto de max_depth ===")
df_depth

---
## 6. LightGBM: Experimentación con Hiperparámetros

In [ ]:
# Datos en formato LightGBM
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Configuración base
params_lgb_base = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'seed': 42
}

# Probamos diferentes learning rates
results_lgb_lr = []

for lr in learning_rates:
    params = params_lgb_base.copy()
    params['learning_rate'] = lr
    model = lgb.train(params, train_data, num_boost_round=100, valid_sets=[test_data],
                      callbacks=[lgb.early_stopping(10), lgb.log_evaluation(0)])
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results_lgb_lr.append({'learning_rate': lr, 'RMSE': rmse, 'best_iter': model.best_iteration})

df_lgb_lr = pd.DataFrame(results_lgb_lr)
print("=== LightGBM: Efecto de learning_rate ===")
df_lgb_lr

In [ ]:
# En LightGBM, num_leaves es análogo a max_depth pero más flexible
num_leaves_values = [7, 15, 31, 63]  # 2^max_depth aproximadamente
results_lgb_leaves = []

for leaves in num_leaves_values:
    params = params_lgb_base.copy()
    params['learning_rate'] = 0.1
    params['num_leaves'] = leaves
    model = lgb.train(params, train_data, num_boost_round=100, valid_sets=[test_data],
                      callbacks=[lgb.early_stopping(10), lgb.log_evaluation(0)])
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results_lgb_leaves.append({'num_leaves': leaves, 'RMSE': rmse, 'best_iter': model.best_iteration})

df_lgb_leaves = pd.DataFrame(results_lgb_leaves)
print("=== LightGBM: Efecto de num_leaves ===")
df_lgb_leaves

---
## 7. Early Stopping en Acción

Observamos cómo el early stopping detiene el entrenamiento cuando el error de validación deja de mejorar.

In [ ]:
# XGBoost con early stopping
params_es = params_base.copy()
params_es['learning_rate'] = 0.1
params_es['max_depth'] = 5

evals_result = {}
model_es = xgb.train(params_es, dtrain, num_boost_round=500, 
                      evals=[(dtrain, 'train'), (dtest, 'test')],
                      early_stopping_rounds=20, verbose_eval=False,
                      evals_result=evals_result)

print(f"Mejor iteración: {model_es.best_iteration}")

# Graficamos la evolución del RMSE
epochs = len(evals_result['train']['rmse'])
x_axis = range(0, epochs)

plt.figure(figsize=(10, 5))
plt.plot(x_axis, evals_result['train']['rmse'], 'b-', label='Train RMSE')
plt.plot(x_axis, evals_result['test']['rmse'], 'r-', label='Test RMSE')
plt.axvline(x=model_es.best_iteration, color='g', linestyle='--', label='Mejor iteración')
plt.xlabel('Iteración')
plt.ylabel('RMSE')
plt.title('Early Stopping: Evolución del RMSE')
plt.legend()
plt.grid(True)
plt.show()

---
## 8. Comparación de Tiempos: XGBoost vs Random Forest

Generamos un dataset mediano y comparamos tiempos de entrenamiento.

In [ ]:
# Generamos dataset más grande
n_large = 50000
n_features = 20
X_large = np.random.randn(n_large, n_features)
y_large = X_large[:, 0] * 2 + X_large[:, 1] * 3 + np.random.randn(n_large) * 0.5

# Random Forest
start = time.time()
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_large, y_large)
time_rf = time.time() - start

# XGBoost
start = time.time()
xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42)
xgb_model.fit(X_large, y_large)
time_xgb = time.time() - start

# LightGBM
start = time.time()
lgb_model = lgb.LGBMRegressor(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42, verbose=-1)
lgb_model.fit(X_large, y_large)
time_lgb = time.time() - start

print(f"Tiempo Random Forest: {time_rf:.2f} segundos")
print(f"Tiempo XGBoost: {time_xgb:.2f} segundos")
print(f"Tiempo LightGBM: {time_lgb:.2f} segundos")

# Gráfico comparativo
plt.figure(figsize=(8, 5))
plt.bar(['Random Forest', 'XGBoost', 'LightGBM'], [time_rf, time_xgb, time_lgb], color=['blue', 'green', 'orange'])
plt.ylabel('Tiempo (segundos)')
plt.title('Comparación de tiempos de entrenamiento')
plt.grid(axis='y')
plt.show()

---
## 9. Experimentación Adicional: Curvas de Entrenamiento y Validación

Mostramos cómo varía el rendimiento con diferentes combinaciones de hiperparámetros.

In [ ]:
# Función para entrenar y registrar histórico
def train_with_history(model_class, params, X_train, y_train, X_val, y_val, n_rounds=200):
    train_errors = []
    val_errors = []
    
    # Inicializar modelo (simplificado para este ejemplo)
    if model_class == xgb.XGBRegressor:
        model = model_class(**params)
        # No tenemos acceso fácil a histórico, usamos evaluación en cada iteración
        # Esta es una simplificación
    else:
        model = model_class(**params)
    
    # Para este ejemplo, usaremos XGBoost con evals_result
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    params_xgb = params.copy()
    params_xgb['objective'] = 'reg:squarederror'
    params_xgb['eval_metric'] = 'rmse'
    
    evals_result = {}
    model = xgb.train(params_xgb, dtrain, num_boost_round=n_rounds,
                       evals=[(dtrain, 'train'), (dval, 'val')],
                       evals_result=evals_result, verbose_eval=False)
    
    return evals_result['train']['rmse'], evals_result['val']['rmse']

# Probamos diferentes learning rates
X_train_sub, X_val, y_train_sub, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)

lr_options = [0.01, 0.05, 0.1, 0.2]
plt.figure(figsize=(12, 8))

for lr in lr_options:
    params = {'learning_rate': lr, 'max_depth': 3}
    train_rmse, val_rmse = train_with_history(xgb.XGBRegressor, params, X_train_sub, y_train_sub, X_val, y_val)
    plt.plot(val_rmse, label=f'LR={lr}')

plt.xlabel('Iteración')
plt.ylabel('RMSE en validación')
plt.title('Efecto de learning_rate en la convergencia')
plt.legend()
plt.grid(True)
plt.show()

---
## 10. Conclusiones

Hemos explorado los fundamentos del gradient boosting y sus implementaciones modernas:

✔️ **Implementación manual**: Construimos boosting desde cero ajustando árboles a los residuos.
✔️ **Mejora iterativa**: Visualizamos cómo la predicción se acerca al valor real con más árboles.
✔️ **XGBoost y LightGBM**: Experimentamos con learning_rate, max_depth, num_leaves.
✔️ **Early stopping**: Observamos cómo detiene el entrenamiento para evitar sobreajuste.
✔️ **Comparación de tiempos**: LightGBM suele ser más rápido que XGBoost, y ambos más rápidos que Random Forest en datasets grandes.

**Lección clave**: El gradient boosting es extremadamente poderoso pero sensible a hiperparámetros. El early stopping y la regularización son esenciales para controlar el sobreajuste.

En el próximo notebook (NB2) aplicaremos estos conceptos a un problema real de predicción de clics o competencia Kaggle.

---
**Fin del Notebook Conceptual - Semana 7**